# ReasonLite — Kaggle GPU edition (v0.1)
Same source, seed, data split sizes, generation protocol and scoring as the original project. No GitHub login, Colab upload or separate dataset upload is required.

**Notebook settings:** GPU T4 (one GPU used even if two are available), Internet On. Keep the notebook private. Run the baseline section first. The SFT section is opt-in; GRPO is not implemented yet.

The pinned packages are installed only if needed. Kaggle's CUDA-enabled PyTorch is retained. No measured GPU results are prefilled.

In [ ]:
import os, sys, subprocess, json, base64, io, zipfile, shutil
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

WORK = Path('/kaggle/working')
assert WORK.is_dir(), 'This notebook is designed for Kaggle.'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'
ROOT = WORK / 'ReasonLite'
# Original source archive embedded below: no additional uploads needed.
SOURCE_ZIP = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ZIP))) as archive:
    for item in archive.infolist():
        if item.is_dir():
            continue
        dest = WORK / item.filename
        assert dest.resolve().is_relative_to(ROOT.resolve())
        payload = archive.read(item)
        if dest.exists():
            assert dest.read_bytes() == payload, f'Existing source differs: {dest}'
        else:
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(payload)
os.chdir(ROOT)

def run(*args):
    subprocess.run([sys.executable, '-u', *map(str,args)], cwd=ROOT, check=True)

def ensure(requirements):
    missing = []
    for req in requirements:
        pkg, expected = req.split('==')
        try:
            actual = version(pkg)
        except PackageNotFoundError:
            actual = None
        if actual != expected:
            missing.append(req)
    if missing:
        run('-m','pip','install','--disable-pip-version-check','--upgrade-strategy','only-if-needed',*missing)
    else:
        print('Required versions already installed; skipping pip.')

run('-c', 'import torch; from packaging.version import Version; assert torch.cuda.is_available(), "Enable GPU in Kaggle settings"; assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), "CUDA PyTorch >=2.6 required"; print(torch.__version__, torch.cuda.get_device_name(), torch.cuda.device_count())')
print('Source ready:', ROOT)

## 1. Install baseline dependencies only
No PyTorch/CUDA reinstall. Training-only dependencies are deferred until you enable SFT. Subprocesses load freshly installed packages without needing a notebook kernel restart.

In [ ]:
BASE_REQ = ['transformers==4.51.3','datasets==3.5.0','huggingface-hub==0.30.2','accelerate==1.6.0']
ensure(BASE_REQ)
run('-c','import torch, transformers, datasets, accelerate; from transformers import AutoTokenizer, AutoModelForCausalLM; print(torch.__version__, transformers.__version__, datasets.__version__, accelerate.__version__)')
run('-m','unittest','discover','-s','tests','-v')

## 2. Freeze data and revisions
If you already prepared a frozen data archive in Colab, add it as a **private Kaggle input** and set `RESTORE_DATA_ZIP` to its full path below. Otherwise leave it blank. The official test is reserved and not evaluated here.

In [ ]:
RESTORE_DATA_ZIP = ''  # optional: /kaggle/input/.../ReasonLite_frozen_data.zip
manifest_path = ROOT/'data/prepared/manifest.json'
if RESTORE_DATA_ZIP and not manifest_path.exists():
    with zipfile.ZipFile(RESTORE_DATA_ZIP) as z:
        for item in z.infolist():
            if item.is_dir(): continue
            dest = ROOT / item.filename
            assert dest.resolve().is_relative_to((ROOT/'data').resolve())
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(z.read(item))
if not manifest_path.exists():
    if manifest_path.parent.exists():
        raise RuntimeError('Incomplete data directory from an interrupted run; inspect it before preparing again.')
    run('-m','reasonlite.data','--include-test')
else:
    print('Reusing frozen dataset and model revision.')
run('-c', 'from reasonlite.data import load_rows; m,_=load_rows("data/prepared","validation"); assert m["seed"]==42; assert {k:v["count"] for k,v in m["files"].items()}=={"sft":512,"rl":256,"validation":128,"test":1319}; [load_rows("data/prepared",s) for s in ("sft","rl","test")]; print(m["model_revision"],m["dataset_revision"])')

## 3. Smoke evaluation (16 questions)
This is real greedy generation, not simulated output. The original protocol uses batch size 1 and a 384-token cap. Run this before the complete 128-question validation. If a run is interrupted, its partial folder is retained; a retry uses a new directory.

In [ ]:
def evaluation(name, limit, adapter=None):
    out = ROOT/'runs'/name
    if out.exists():
        if (out/'metrics.json').exists():
            print('Reusing completed run:', out)
            return out
        out = out.with_name(out.name+'_retry_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
    args = ['-m','reasonlite.evaluate','--device','cuda','--limit',str(limit),'--max-new-tokens','384','--out',str(out)]
    if adapter is not None: args.extend(['--adapter',str(adapter)])
    run(*args)
    return out
smoke = evaluation('e0_smoke',16)
print((smoke/'metrics.json').read_text())

## 4. Full development baseline (128 questions)
This is the comparison baseline for later SFT. It can take substantially longer than the smoke run. Do not reduce its size and label it as the full validation result.

In [ ]:
baseline = evaluation('e0_validation',128)
print((baseline/'metrics.json').read_text())

## 5. Export evidence — download now
Use Kaggle's Files/Output panel to download these ZIP files before ending the session. If saving a notebook version, use the option that preserves the current outputs when available; a full rerun may repeat GPU computation. Keep raw data private. The evidence ZIP includes the smoke and full baseline predictions and metrics; the frozen data ZIP preserves the exact split manifest.

In [ ]:
def export_baseline():
    assert (baseline/'metrics.json').exists()
    from IPython.display import FileLink, display
    for name, folder in [('ReasonLite_baseline_evidence','runs'),('ReasonLite_frozen_data','data')]:
        file = shutil.make_archive(str(WORK/name),'zip',str(ROOT),folder)
        print(file)
        display(FileLink(file))
export_baseline()

## 6. Optional SFT (next sprint)
Set `RUN_SFT=True` only when you are ready for training. A two-step smoke run checks the actual trainer, completion mask and GPU before the full 512-example experiment. Keep the same GPU type and package versions for paired evaluation. If the smoke fails, stop and inspect its traceback. No GRPO training is included.

In [ ]:
RUN_SFT = False
if RUN_SFT:
    ensure(['trl==0.17.0','peft==0.15.2']+BASE_REQ)
    run('-c','from trl import SFTConfig,SFTTrainer; from peft import LoraConfig; print("Training imports OK")')
    def train(name, steps=None):
        out = ROOT/'runs'/name
        if out.exists():
            if (out/'train_metrics.json').exists() and (out/'adapter/reasonlite_training.json').exists():
                return out
            out = out.with_name(out.name+'_retry_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
        args = ['-m','reasonlite.train_sft','--out',str(out)]
        if steps is not None: args += ['--max-steps',str(steps)]
        run(*args)
        return out
    sft_smoke = train('sft_smoke',2)
    print((sft_smoke/'train_metrics.json').read_text())
else:
    print('SFT disabled; baseline-only mode.')

In [ ]:
if RUN_SFT:
    sft = train('sft')
    e1 = evaluation('e1_validation',128,adapter=sft/'adapter')
    run('-m','reasonlite.compare',str(baseline),str(e1),'--out',str(ROOT/'runs/e0_vs_e1.json'))
    # Preserve compact adapter, logs and comparisons, not redundant optimizer checkpoints.
    out = WORK/'ReasonLite_sft_evidence.zip'
    with zipfile.ZipFile(out,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for file in (ROOT/'runs').rglob('*'):
            if file.is_file() and 'checkpoints' not in file.relative_to(ROOT/'runs').parts:
                z.write(file,file.relative_to(ROOT))
    print(out)
    from IPython.display import FileLink, display
    display(FileLink(str(out)))